# RetailPulse: Data Cleaning and Preparation

## Overview

This notebook focuses on cleaning and preparing the Amazon Sale Report dataset for reliable analysis. It applies data quality findings from the previous notebook to address missing values, inconsistent categorical data, data types, and other identified issues.

## Objectives

* Create a working copy of the raw dataset for cleaning.
* Remove columns identified as unnecessary for the planned analysis.
* Handle missing values using appropriate strategies.
* Standardize categorical values and correct data types where needed.
* Investigate unusual values and validate the cleaned dataset.
* Save the cleaned dataset for subsequent analysis.

## Approach

The original raw dataset will remain unchanged. All cleaning operations will be performed on a separate working DataFrame, and the results will be validated before saving the processed dataset.


## 1. Load the Dataset

The raw Amazon Sale Report dataset is loaded into a Pandas DataFrame. A separate working copy is created to ensure that all cleaning operations leave the original dataset unchanged.


In [88]:
import pandas as pd
import numpy as np

csv_file = "../data/raw/Amazon Sale Report.csv"

df = pd.read_csv(csv_file)

df_cleaned = df.copy()

print("Dataset loaded successfully.")
print(f"Original dataset shape: {df.shape}")
print(f"Working copy shape: {df_cleaned.shape}")

Dataset loaded successfully.
Original dataset shape: (128975, 24)
Working copy shape: (128975, 24)


C:\Users\praya\AppData\Local\Temp\ipykernel_12844\4064105777.py:6: DtypeWarning: Columns (23: Unnamed: 22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_file)


## 2. Standardize Column Names

This section standardizes column names by removing leading and trailing whitespace and replacing spaces with underscores. This creates consistent, easy-to-reference column names for subsequent cleaning and analysis.


In [89]:
df_cleaned.columns = (
    df_cleaned.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("Columns names are standardized successfully.")
df_cleaned.columns.to_list()

Columns names are standardized successfully.


['index',
 'order_id',
 'date',
 'status',
 'fulfilment',
 'sales_channel',
 'ship_service_level',
 'style',
 'sku',
 'category',
 'size',
 'asin',
 'courier_status',
 'qty',
 'currency',
 'amount',
 'ship_city',
 'ship_state',
 'ship_postal_code',
 'ship_country',
 'promotion_ids',
 'b2b',
 'fulfilled_by',
 'unnamed:_22']

## 3. Remove Unnecessary Columns

This section removes columns identified during the data quality assessment as unnecessary for the planned analysis. The selected columns are removed from the working DataFrame, while the original raw dataset remains unchanged.


In [90]:
drop_columns = [
    "index",
    "currency",
    "ship_city",
    "ship_postal_code",
    "promotion_ids",
    "unnamed:_22"
]

drop_columns

['index',
 'currency',
 'ship_city',
 'ship_postal_code',
 'promotion_ids',
 'unnamed:_22']

In [91]:
df_cleaned.drop(columns=drop_columns, inplace=True)

In [92]:
print(f"Columns removed: {len(drop_columns)}")
print(f"Remaining columns: {df_cleaned.shape[1]}")

df_cleaned.columns.to_list()

Columns removed: 6
Remaining columns: 18


['order_id',
 'date',
 'status',
 'fulfilment',
 'sales_channel',
 'ship_service_level',
 'style',
 'sku',
 'category',
 'size',
 'asin',
 'courier_status',
 'qty',
 'amount',
 'ship_state',
 'ship_country',
 'b2b',
 'fulfilled_by']

## 4. Investigate Duplicate Order IDs

An order ID may appear in multiple rows when an order contains multiple products or line items. Therefore, repeated order IDs do not necessarily indicate duplicate records.

This section examines repeated order IDs and checks for exact duplicate rows. Repeated order IDs will be retained because they may represent valid sales line items, while exact duplicate records will be removed to prevent duplicate entries from affecting the analysis.

In [93]:
print("Duplicate order ids: ", df_cleaned["order_id"].duplicated().sum())

print("Exact duplicate records along dataset: ", df_cleaned.duplicated().sum())


Duplicate order ids:  8597
Exact duplicate records along dataset:  6


In [94]:
duplicates_before = df_cleaned.duplicated().sum()

df_cleaned.drop_duplicates(inplace=True)

duplicates_after = df_cleaned.duplicated().sum()

print(f"Exact duplicate records removed: {duplicates_before - duplicates_after}")
print(f"Exact duplicate records remaining: {duplicates_after}")
print(f"Updated dataset shape: {df_cleaned.shape}")

Exact duplicate records removed: 6
Exact duplicate records remaining: 0
Updated dataset shape: (128969, 18)


## 5. Convert the Date Column

The `date` column represents the date associated with each sales record. Converting it to a datetime data type enables accurate date-based filtering, time-series analysis, and the calculation of daily, monthly, and yearly sales trends.

This section converts the date values to a datetime format and checks for any values that could not be converted successfully.


In [95]:
df_cleaned["date"] = pd.to_datetime(
    df_cleaned["date"],
    format="%m-%d-%y",
    errors="coerce"
)

print("Date column converted successfully.")
print(f"Data type: {df_cleaned['date'].dtype}")
print(f"Missing dates: {df_cleaned['date'].isna().sum()}")
print(f"Date range: {df_cleaned['date'].min()} to {df_cleaned['date'].max()}")

Date column converted successfully.
Data type: datetime64[us]
Missing dates: 0
Date range: 2022-03-31 00:00:00 to 2022-06-29 00:00:00


## 6. Handle Missing Courier Status Values

The `courier_status` column contains missing values that need to be addressed for consistent shipment analysis. These missing values will be replaced with `Unshipped` to provide a consistent category for subsequent analysis.


In [123]:
df_cleaned["courier_status"] = df_cleaned["courier_status"].fillna("Unshipped")

print(f"Missing values after cleaning: {df_cleaned["courier_status"].isna().sum()}")
print("\nCourier status value counts:")
print(df_cleaned["courier_status"].value_counts())

Missing values after cleaning: 0

Courier status value counts:
courier_status
Shipped      109484
Unshipped     13553
Cancelled      5932
Name: count, dtype: int64


## 7. Handle Missing Amount Values

The `amount` column represents the monetary value associated with each sales record. Missing values are replaced with `0` to maintain consistent data for subsequent calculations and analysis.

This approach assumes that missing amount values should be treated as zero for this project. Missing values and genuine zero-value transactions will be considered separately during the final inspection.


In [124]:
df_cleaned["amount"] = df_cleaned["amount"].fillna("0")

print(f"Missing values after cleaning: {df_cleaned["amount"].isna().sum()}")
print("\nAmount value counts:")
print(df_cleaned["amount"].value_counts())

Missing values after cleaning: 0

Amount value counts:
amount
0         7792
399.0     5442
771.0     2797
735.0     2436
0.0       2343
          ... 
719.05       1
829.52       1
708.58       1
1244.0       1
639.0        1
Name: count, Length: 1411, dtype: int64


In [227]:
df_cleaned["amount"] = pd.to_numeric(df_cleaned["amount"])

## 8. Handle Missing Shipping Country Values

The `ship_country` column identifies the destination country for each sales record. During the data quality assessment, all non-missing values were found to be `IN`, while some records contained missing values.

Since the available records consistently identify India as the shipping country, missing values will be replaced with `IN` to maintain consistency in the dataset.


In [ ]:
df_cleaned["ship_country"] = df_cleaned["ship_country"].fillna("IN")

print(f"Missing values after cleaning: {df_cleaned["ship_country"].isna().sum()}")
print("\nShipping Country value counts:")
print(df_cleaned["ship_country"].value_counts())

Missing values after cleaning: 0

Shipping Country value counts:
ship_country
IN    128969
Name: count, dtype: int64


## 9. Handle Missing Fulfilment Values

The `fulfilled_by` column identifies the fulfilment method associated with a sales record. The data quality assessment found that a substantial proportion of values are missing, while the available records contain the category `Easy Ship`.

Since the missing values cannot be reliably assigned to a known fulfilment method, they will be replaced with `Unknown`. This preserves the existing information while clearly identifying records with unavailable fulfilment details.


In [135]:
df_cleaned["fulfilled_by"] = df_cleaned["fulfilled_by"].fillna("Unknown")

print(f"Missing values after cleaning: {df_cleaned["fulfilled_by"].isna().sum()}")
print("\nFullfilment values countd:")
print(df_cleaned["fulfilled_by"].value_counts())

Missing values after cleaning: 0

Fullfilment values countd:
fulfilled_by
Unknown      89692
Easy Ship    39277
Name: count, dtype: int64


## 10. Standardize B2B Values

The `b2b` column identifies whether a sales record is associated with a business-to-business (B2B) transaction. Its values are currently represented as Boolean values, `True` and `False`.

To improve readability and make the categories easier to interpret during analysis, `True` will be replaced with `Business` and `False` with `Customer`.


In [148]:
df_cleaned["b2b"] = df_cleaned["b2b"].replace({True: "Business", False: "Customer"})

print("B2B values standardized successfully.")
print("\nB2B value counts:")
print(df_cleaned["b2b"].value_counts())

B2B values standardized successfully.

B2B value counts:
b2b
Customer    128098
Business       871
Name: count, dtype: int64


## 11. Clean Shipping State Values

The data quality assessment identified inconsistencies in the `ship_state` column, including leading or trailing whitespace, inconsistent capitalization, missing values, abbreviations, and alternative spellings of state names.

To improve consistency for geographical analysis, the state names will be standardized by removing unnecessary whitespace and converting values to uppercase. Missing values will be replaced with `Unknown` because the correct state cannot be determined from this column alone.

A state variants dictionary will then be used to merge recognized abbreviations, spelling variations, and alternative names into consistent categories. The number of unique state values and their distribution will be checked after standardization.


In [191]:
df_cleaned["ship_state"] = (
    df_cleaned["ship_state"]
    .str.strip()
    .str.upper()
    .fillna("Unknown")
)

print(f"Missing values after cleaning: {df_cleaned["ship_state"].isna().sum()}")
print(f"Unique state values after cleaning: {df_cleaned["ship_state"].nunique()}")

print("\nshipping states:")
df_cleaned["ship_state"].value_counts().sort_index()

Missing values after cleaning: 0
Unique state values after cleaning: 48

shipping states:


ship_state
ANDAMAN & NICOBAR           257
ANDHRA PRADESH             5430
APO                           1
AR                            1
ARUNACHAL PRADESH           146
ASSAM                      1663
BIHAR                      2114
CHANDIGARH                  333
CHHATTISGARH                909
DADRA AND NAGAR              70
DELHI                      6967
GOA                        1137
GUJARAT                    4489
HARYANA                    4415
HIMACHAL PRADESH            788
JAMMU & KASHMIR             702
JHARKHAND                  1456
KARNATAKA                 17325
KERALA                     6584
LADAKH                       43
LAKSHADWEEP                   4
MADHYA PRADESH             2529
MAHARASHTRA               22259
MANIPUR                     316
MEGHALAYA                   207
MIZORAM                      76
NAGALAND                    185
NEW DELHI                    81
NL                            2
ODISHA                     2136
ORISSA                       

In [ ]:
state_variants = {
    "ARUNACHAL PRADESH": "ARUNACHAL PRADESH",
    "AR": "ARUNACHAL PRADESH",
    "DELHI": "DELHI",
    "NEW DELHI": "DELHI",
    "NAGALAND": "NAGALAND",
    "NL": "NAGALAND",
    "ODISHA": "ODISHA",
    "ORISSA": "ODISHA",
    "PUDUCHERRY": "PUDUCHERRY",
    "PONDICHERRY": "PUDUCHERRY",
    "PUNJAB": "PUNJAB",
    "PB": "PUNJAB",
    "PUNJAB/MOHALI/ZIRAKPUR": "PUNJAB",
    "RAJASTHAN": "RAJASTHAN",
    "RAJSHTHAN": "RAJASTHAN",
    "RAJSTHAN": "RAJASTHAN",
    "RJ": "RAJASTHAN"
}

state_variants

{'ARUNACHAL PRADESH': 'ARUNACHAL PRADESH',
 'AR': 'ARUNACHAL PRADESH',
 'DELHI': 'DELHI',
 'NEW DELHI': 'DELHI',
 'NAGALAND': 'NAGALAND',
 'NL': 'NAGALAND',
 'ODISHA': 'ODISHA',
 'ORISSA': 'ODISHA',
 'PUDUCHERRY': 'PUDUCHERRY',
 'PONDICHERRY': 'PUDUCHERRY',
 'PUNJAB': 'PUNJAB',
 'PB': 'PUNJAB',
 'PUNJAB/MOHALI/ZIRAKPUR': 'PUNJAB',
 'RAJASTHAN': 'RAJASTHAN',
 'RAJSHTHAN': 'RAJASTHAN',
 'RAJSTHAN': 'RAJASTHAN',
 'RJ': 'RAJASTHAN'}

In [209]:
df_cleaned["ship_state"] = df_cleaned["ship_state"].replace(state_variants)

print("State variants merged successfully.")
print(f"Unique state values after cleaning: {df_cleaned['ship_state'].nunique()}")

print("\nUpdated state value counts:")
df_cleaned["ship_state"].value_counts().sort_index()

State variants merged successfully.
Unique state values after cleaning: 38

Updated state value counts:


ship_state
ANDAMAN & NICOBAR      257
ANDHRA PRADESH        5430
APO                      1
ARUNACHAL PRADESH      147
ASSAM                 1663
BIHAR                 2114
CHANDIGARH             333
CHHATTISGARH           909
DADRA AND NAGAR         70
DELHI                 7048
GOA                   1137
GUJARAT               4489
HARYANA               4415
HIMACHAL PRADESH       788
JAMMU & KASHMIR        702
JHARKHAND             1456
KARNATAKA            17325
KERALA                6584
LADAKH                  43
LAKSHADWEEP              4
MADHYA PRADESH        2529
MAHARASHTRA          22259
MANIPUR                316
MEGHALAYA              207
MIZORAM                 76
NAGALAND               187
ODISHA                2139
PUDUCHERRY             351
PUNJAB                1918
RAJASTHAN             2718
SIKKIM                 205
TAMIL NADU           11483
TELANGANA            11330
TRIPURA                151
UNKNOWN                 33
UTTAR PRADESH        10637
UTTARAKHAND      

## 12. Final Data Validation

This section verifies the results of the data cleaning process and checks whether the dataset is ready for further analysis.

The validation covers:

* Dataset dimensions and column names.
* Data types and missing values.
* Exact duplicate records.
* Consistency of cleaned categorical values.
* Validity of date and numeric columns.

The results will be reviewed to identify any remaining issues before the cleaned dataset is saved for feature engineering and exploratory data analysis.


### 12.1 Check the Dataset Structure

This check reviews the cleaned dataset's dimensions, column names, data types, and sample records. It helps confirm that the expected columns remain and that the cleaning operations have not unexpectedly changed the dataset's structure.


In [216]:
print("Dataset Shape:", df_cleaned.shape)

print("\nColumn Names:")
print(df_cleaned.columns.to_list())

print("\nData Types:")
print(df_cleaned.dtypes)

print("\nFirst 10 Records:")
display(df_cleaned.head(10))

Dataset Shape: (128969, 18)

Column Names:
['order_id', 'date', 'status', 'fulfilment', 'sales_channel', 'ship_service_level', 'style', 'sku', 'category', 'size', 'asin', 'courier_status', 'qty', 'amount', 'ship_state', 'ship_country', 'b2b', 'fulfilled_by']

Data Types:
order_id                         str
date                  datetime64[us]
status                           str
fulfilment                       str
sales_channel                    str
ship_service_level               str
style                            str
sku                              str
category                         str
size                             str
asin                             str
courier_status                   str
qty                            int64
amount                        object
ship_state                       str
ship_country                     str
b2b                           object
fulfilled_by                     str
dtype: object

First 10 Records:


,order_id,date,status,fulfilment,sales_channel,ship_service_level,style,sku,category,size,asin,courier_status,qty,amount,ship_state,ship_country,b2b,fulfilled_by
0,405-8078784-5731545,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,B09KXVBD7Z,Unshipped,0,647.62,MAHARASHTRA,IN,Customer,Easy Ship
1,171-9198151-1101146,2022-04-30,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,3XL,B09K3WFS32,Shipped,1,406.0,KARNATAKA,IN,Customer,Easy Ship
2,404-0687676-7273146,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,XL,B07WV4JV4D,Shipped,1,329.0,MAHARASHTRA,IN,Business,Unknown
3,403-9615377-8133951,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,L,B099NRCT7B,Unshipped,0,753.33,PUDUCHERRY,IN,Customer,Easy Ship
4,407-1069790-7240320,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,3XL,B098714BZP,Shipped,1,574.0,TAMIL NADU,IN,Customer,Unknown
5,404-1490984-4578765,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,SET264,SET264-KR-NP-XL,Set,XL,B08YN7XDSG,Shipped,1,824.0,UTTAR PRADESH,IN,Customer,Unknown
6,408-5748499-6859555,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,J0095,J0095-SET-L,Set,L,B08CMHNWBN,Shipped,1,653.0,CHANDIGARH,IN,Customer,Unknown
7,406-7807733-3785945,2022-04-30,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3405,JNE3405-KR-S,kurta,S,B081WX4G4Q,Shipped,1,399.0,TELANGANA,IN,Customer,Easy Ship
8,407-5443024-5233168,2022-04-30,Cancelled,Amazon,Amazon.in,Expedited,SET200,SET200-KR-NP-A-XXXL,Set,3XL,B08L91ZZXN,Cancelled,0,0,TELANGANA,IN,Customer,Unknown
9,402-4393761-0311520,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3461,JNE3461-KR-XXL,kurta,XXL,B08B3XF5MH,Shipped,1,363.0,TAMIL NADU,IN,Customer,Unknown


### 12.2 Check Missing Values

This check calculates the number and percentage of missing values in each column. It verifies whether the cleaning operations have addressed the identified missing values and helps highlight any remaining gaps in the dataset.


In [211]:
missing_summary = pd.DataFrame({
    "Missing Values": df_cleaned.isna().sum(),
    "Missing Percentage": (df_cleaned.isna().mean() * 100).round(2)
})

missing_summary = missing_summary.sort_values(
    by="Missing Values",
    ascending=False
)

display(missing_summary)

,Missing Values,Missing Percentage
order_id,0,0.0
date,0,0.0
status,0,0.0
fulfilment,0,0.0
sales_channel,0,0.0
ship_service_level,0,0.0
style,0,0.0
sku,0,0.0
category,0,0.0
size,0,0.0


### 12.3 Check Duplicate Records

This check verifies that no exact duplicate rows remain after the duplicate-removal step. It also reports the total number of records in the cleaned dataset to confirm the final row count.


In [212]:
print("Exact duplicate records:", df_cleaned.duplicated().sum())
print("Total records:", len(df_cleaned))

Exact duplicate records: 0
Total records: 128969


### 12.4 Validate Cleaned Categorical Columns

This check reviews the value distributions of selected categorical columns after cleaning. It verifies that the intended categories are present and helps identify unexpected values or inconsistencies.

The review covers courier status, shipping country, fulfilment method, B2B category, and shipping state.


In [215]:
print("Courier Status:")
display(df_cleaned["courier_status"].value_counts())

print("Shipping Country:")
display(df_cleaned["ship_country"].value_counts())

print("Fulfilment Method:")
display(df_cleaned["fulfilled_by"].value_counts())

print("B2B Categories:")
display(df_cleaned["b2b"].value_counts())

print("Shipping States:")
display(df_cleaned["ship_state"].value_counts().sort_index())

Courier Status:


courier_status
Shipped      109484
Unshipped     13553
Cancelled      5932
Name: count, dtype: int64

Shipping Country:


ship_country
IN    128969
Name: count, dtype: int64

Fulfilment Method:


fulfilled_by
Unknown      89692
Easy Ship    39277
Name: count, dtype: int64

B2B Categories:


b2b
Customer    128098
Business       871
Name: count, dtype: int64

Shipping States:


ship_state
ANDAMAN & NICOBAR      257
ANDHRA PRADESH        5430
APO                      1
ARUNACHAL PRADESH      147
ASSAM                 1663
BIHAR                 2114
CHANDIGARH             333
CHHATTISGARH           909
DADRA AND NAGAR         70
DELHI                 7048
GOA                   1137
GUJARAT               4489
HARYANA               4415
HIMACHAL PRADESH       788
JAMMU & KASHMIR        702
JHARKHAND             1456
KARNATAKA            17325
KERALA                6584
LADAKH                  43
LAKSHADWEEP              4
MADHYA PRADESH        2529
MAHARASHTRA          22259
MANIPUR                316
MEGHALAYA              207
MIZORAM                 76
NAGALAND               187
ODISHA                2139
PUDUCHERRY             351
PUNJAB                1918
RAJASTHAN             2718
SIKKIM                 205
TAMIL NADU           11483
TELANGANA            11330
TRIPURA                151
UNKNOWN                 33
UTTAR PRADESH        10637
UTTARAKHAND      

### 12.5 Validate Dates and Numeric Columns

This check verifies the data types and ranges of the date, quantity, and amount columns. It reviews the earliest and latest dates, checks for missing dates and negative quantities or amounts, and counts zero-value amounts.

These checks help identify potential issues before the dataset is used for feature engineering and exploratory data analysis.


In [224]:
print("Date \nData Type:", df_cleaned["date"].dtype)
print("Earliest Date:", df_cleaned["date"].min())
print("Latest Date:", df_cleaned["date"].max())
print("Missing Dates:", df_cleaned["date"].isna().sum())

print("\nQuantity \nData Type:", df_cleaned["qty"].dtype)
print("Negative Quantities:", (df_cleaned["qty"] < 0).sum())

print("\nAmount \nData Type:", df_cleaned["amount"].dtype)
print("Negative Amounts:", (df_cleaned["amount"] < 0).sum())
print("Zero Amounts:", (df_cleaned["amount"] == 0).sum())

Date 
Data Type: datetime64[us]
Earliest Date: 2022-03-31 00:00:00
Latest Date: 2022-06-29 00:00:00
Missing Dates: 0

Quantity 
Data Type: int64
Negative Quantities: 0

Amount 
Data Type: float64
Negative Amounts: 0
Zero Amounts: 10135


### 13. Save the Cleaned Dataset

After completing the cleaning and validation steps, the cleaned dataset will be exported to a CSV file in the `data/processed/` directory.

Saving the processed dataset allows subsequent stages of the RetailPulse project to use the cleaned data directly, ensuring a consistent and reproducible workflow without modifying the original raw dataset.


In [230]:
cleaned_csv = "../data/processed/cleaned_dataset.csv"

df_cleaned.to_csv(cleaned_csv, index=False)

print("Cleaned dataset exported successfully.")
print(f"File location: {cleaned_csv}")

Cleaned dataset exported successfully.
File location: ../data/processed/cleaned_dataset.csv


## Conclusion

The data cleaning process is complete. The dataset has been standardized, exact duplicate records have been removed, missing values have been handled according to the documented assumptions, and selected categorical values have been cleaned for consistency.

Final validation checks have been performed on the dataset structure, missing values, duplicate records, categorical columns, dates, and numeric columns.

The cleaned dataset is now ready for the next stage of the RetailPulse project: **feature engineering**, where additional analytical columns will be created to support sales analysis and business insights.
